# 02. Derive trial metrics

Turns the event log into one row per trial using the proposal's definitions (3.2-3.3). **Dummy data by default.**

- A *selection* is a tap on a menu item; it is an **error** if the item is not on the correct path P*.
- An *error episode* starts at an error and ends when the current node is back on P* (an on-path tap, or a Back/collapse landing on-path); if the trial ends first, it ends at the trial end.
- **R** = sum of episode durations (Eq. 1); **T_find** = time of the terminal event; **T_clean = T_find - R** (Eq. 2).
- Give-up and time-out trials keep T_find = trial end but `success = 0`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

DATA_DIR = ROOT / "data" / "dummy"      # <- point this at data/raw/<your export folder> for real data
OUT_DIR = ROOT / "outputs"

from pipeline.prepare import prepare

prep = prepare(DATA_DIR)
trials, episodes = prep.trials, prep.episodes
trials.head()

## Sanity checks
Eq. 2 must hold exactly, and the logged `on_path` flag should agree with the tree.

In [ ]:
import numpy as np
print('T_find = T_clean + R for every trial:', bool(np.allclose(trials.T_find, trials.T_clean + trials.R)))
print('trials:', len(trials), '| with errors:', int((trials.E > 0).sum()), '| failed:', int((trials.success == 0).sum()))
print('validation: logged on_path mismatches =', 0 if 'on_path_matches_tree' not in prep.report.failed_names else 'see notebook 01')

## Look at one trial
Change `TRIAL_ID`. The table shows each tap and whether it was on the correct path; the episodes below it are what R is built from.

In [ ]:
TRIAL_ID = trials[trials.E > 0].trial_id.iloc[0]
tree = prep.clean.tree
ev = prep.clean.events[prep.clean.events.trial_id == TRIAL_ID][['t_ms', 'event_type', 'node_id', 'current_node_after']].copy()
target = trials.loc[trials.trial_id == TRIAL_ID, 'target_node'].iloc[0]
ev['on_path_now'] = ev.current_node_after.isin(tree.path(target))
print('correct path:', ' > '.join(tree.path(target)))
ev

In [ ]:
print(trials.loc[trials.trial_id == TRIAL_ID, ['trial_id', 'pattern', 'E', 'R', 'T_find', 'T_clean', 'selection_path']].to_string(index=False))
episodes[episodes.trial_id == TRIAL_ID]

## Check against the generator's ground truth (dummy data only)
The dummy generator records the true E, R and T_find for each trial; the pipeline must reproduce them exactly.

In [ ]:
gt_path = DATA_DIR / '_ground_truth_trials.csv'
if gt_path.exists():
    gt = pd.read_csv(gt_path)
    m = trials.merge(gt, on='trial_id', suffixes=('', '_true'))
    print('trials compared:', len(m))
    print('E equal:', bool((m.E == m.E_true).all()), '| R equal:', bool(np.allclose(m.R, m.R_true)), '| T_find equal:', bool(np.allclose(m.T_find, m.T_find_true)))
else:
    print('No ground-truth file (expected for real data).')